# Day 071 — Exercise 3: index_images

**What you'll build:** `index_images(images_with_ids, describe_fn, embed_fn) -> ImageIndex` — the batch pipeline that describes and embeds a collection of images.

**Why it matters:** This is the offline indexing step. Run it once on your image collection; then search it instantly any number of times.

In [ ]:
import base64, hashlib, io
import numpy as np
from PIL import Image

_SEARCH_PROMPT = (
    'Describe this image in detail for use in a semantic search index. '
    'Include: main subjects, colors, textures, setting, and visible text. '
    'Write one concise paragraph of 2-3 sentences.'
)

def image_to_base64(img, format='PNG'):
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

def cosine_similarity(a, b):
    va = np.array(a, dtype=np.float32)
    vb = np.array(b, dtype=np.float32)
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom == 0.0:
        return 0.0
    return float(np.dot(va, vb) / denom)

class ImageIndex:
    def __init__(self):
        self._items = []
    def add(self, image_id, description, embedding, metadata=None):
        self._items.append({'id': image_id, 'description': description,
                            'embedding': np.array(embedding, dtype=np.float32),
                            'metadata': metadata or {}})
    def search(self, query_embedding, n=5):
        if not self._items:
            return []
        q = np.array(query_embedding, dtype=np.float32)
        scored = [(cosine_similarity(q, item['embedding']), item) for item in self._items]
        scored.sort(key=lambda x: -x[0])
        top = scored[:min(n, len(scored))]
        return [{'id': item['id'], 'description': item['description'],
                 'score': float(score), 'metadata': item['metadata']}
                for score, item in top]
    def __len__(self):
        return len(self._items)

def describe_image_for_search(img, describe_fn=None):
    img_b64 = image_to_base64(img)
    if describe_fn is not None:
        return describe_fn(img_b64, _SEARCH_PROMPT)
    import ollama
    resp = ollama.chat(model='llava',
                       messages=[{'role': 'user', 'content': _SEARCH_PROMPT, 'images': [img_b64]}])
    return resp['message']['content'].strip()

def embed_text(text, embed_fn=None):
    if embed_fn is not None:
        return embed_fn(text)
    import ollama
    resp = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return resp['embedding']

def index_images(images_with_ids, describe_fn=None, embed_fn=None):
    index = ImageIndex()
    for image_id, img, metadata in images_with_ids:
        desc = describe_image_for_search(img, describe_fn=describe_fn)
        emb  = embed_text(desc, embed_fn=embed_fn)
        index.add(image_id, desc, emb, metadata or {})
    return index

def search_by_text(query, index, embed_fn=None, n=5):
    q_emb = embed_text(query, embed_fn=embed_fn)
    return index.search(q_emb, n=n)

def search_by_image(img, index, describe_fn=None, embed_fn=None, n=5):
    desc  = describe_image_for_search(img, describe_fn=describe_fn)
    q_emb = embed_text(desc, embed_fn=embed_fn)
    return index.search(q_emb, n=n)

import hashlib
def _mock_describe(img_b64, prompt):
    h = int(hashlib.md5(img_b64.encode()).hexdigest()[:4], 16)
    labels = ['a red apple on a table', 'a blue ocean wave',
              'a green forest path', 'a yellow sunflower field']
    return labels[h % len(labels)]

def _mock_embed(text):
    h = int(hashlib.md5(text.encode()).hexdigest()[:8], 16)
    return [((h >> (i * 8)) & 0xff) / 128.0 - 1.0 for i in range(4)]


## Task

Implement `index_images`:

1. Create `index = ImageIndex()`
2. `for image_id, img, metadata in images_with_ids:`
   - `desc = describe_image_for_search(img, describe_fn=describe_fn)`
   - `emb = embed_text(desc, embed_fn=embed_fn)`
   - `index.add(image_id, desc, emb, metadata or {})`
3. Return `index`

## Your Implementation

In [ ]:
def index_images(images_with_ids: list,
                 describe_fn=None,
                 embed_fn=None) -> ImageIndex:
    """Describe, embed, and index a batch of images.

    Args:
        images_with_ids: list of (image_id, img, metadata) tuples
        describe_fn:     callable(img_b64, prompt) -> str for testing
        embed_fn:        callable(text) -> list[float] for testing
    Returns:
        Populated ImageIndex
    """
    raise NotImplementedError


In [ ]:
def index_images(images_with_ids, describe_fn=None, embed_fn=None):
    index = ImageIndex()
    for image_id, img, metadata in images_with_ids:
        desc = describe_image_for_search(img, describe_fn=describe_fn)
        emb  = embed_text(desc, embed_fn=embed_fn)
        index.add(image_id, desc, emb, metadata or {})
    return index


## Automated checks

In [ ]:
score, total = 0, 5
try:
    imgs = [
        ('img_r', Image.new('RGB', (16,16), (220, 50, 50)), {'tag': 'red'}),
        ('img_b', Image.new('RGB', (16,16), (50, 100, 220)), {'tag': 'blue'}),
        ('img_g', Image.new('RGB', (16,16), (50, 180, 80)), {'tag': 'green'}),
    ]

    idx = index_images(imgs, describe_fn=_mock_describe, embed_fn=_mock_embed)

    # returns ImageIndex with 3 items
    assert isinstance(idx, ImageIndex)
    assert len(idx) == 3
    score += 1; print("\u2705 index_images returns ImageIndex with correct item count")

    # search returns dicts with expected keys
    results = idx.search([0.5, 0.5, -0.5, -0.5], n=3)
    assert len(results) == 3
    assert all('id' in r and 'description' in r and 'score' in r and 'metadata' in r
               for r in results)
    score += 1; print("\u2705 indexed items have all required keys")

    # descriptions are non-empty strings
    assert all(isinstance(r['description'], str) and len(r['description']) > 0
               for r in results)
    score += 1; print("\u2705 all descriptions are non-empty strings")

    # metadata is preserved correctly
    ids_in_results = {r['id'] for r in results}
    assert ids_in_results == {'img_r', 'img_b', 'img_g'}
    tags = {r['id']: r['metadata'].get('tag') for r in results}
    assert tags['img_r'] == 'red' and tags['img_b'] == 'blue'
    score += 1; print("\u2705 metadata preserved per image_id")

    # results sorted descending by score
    scores = [r['score'] for r in results]
    assert scores == sorted(scores, reverse=True), f"Not sorted: {scores}"
    score += 1; print("\u2705 results sorted by score descending")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def index_images(images_with_ids, describe_fn=None, embed_fn=None):
    index = ImageIndex()
    for image_id, img, metadata in images_with_ids:
        desc = describe_image_for_search(img, describe_fn=describe_fn)
        emb  = embed_text(desc, embed_fn=embed_fn)
        index.add(image_id, desc, emb, metadata or {})
    return index
```

**Why `metadata or {}`?** If the caller passes `None` as metadata (or omits it), storing `None` in the index would cause `KeyError` when result dicts are built. The `or {}` guard makes the index robust to caller mistakes.

</details>